# Real-Data EEG Pipeline — Patient XU, 7 Hz DBS

Applies the full 7-method DBS removal comparison + ICA pipeline to real clinical recordings.

## Recordings
| File | Condition | sfreq | DBS |
|------|-----------|-------|-----|
| `XUAWAKE7_deidentified.edf`  | Awake, DBS ON  | 256 Hz | 7 Hz |
| `XUSLEEP7_deidentified.edf`  | Sleep,  DBS ON  | 256 Hz | 7 Hz |
| `XUAWAKEPRE_deidentified.edf`| Awake, DBS OFF (baseline) | 200 Hz | — |
| `XUSLEEP_deidentified.edf`   | Sleep,  DBS OFF (baseline) | 200 Hz | — |

The PRE recordings are from a different session — used as the best available brain-only reference.
They are resampled from 200 → 256 Hz for comparison.

## Pipeline
1. Load + standardise to 19-channel 10-20 montage
2. Resample PRE to 256 Hz, bandpass 1–119 Hz, average reference
3. **7-method DBS removal comparison** on AWAKE7 and SLEEP7
4. Choose best method → run ICA (eye + muscle removal)
5. Final comprehensive comparison figure

In [1]:
import sys, warnings, pathlib
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import signal
import mne
mne.set_log_level('WARNING')

from src.filters import ArtifactFilterFactory
from src.preprocessing import EEGPreprocessor

pathlib.Path('../figures/real').mkdir(parents=True, exist_ok=True)

DATA_DIR = pathlib.Path('../data/raw/XU')
DBS_FREQ  = 7.0
TARGET_SFREQ = 256.0

print(f'MNE {mne.__version__}  |  NumPy {np.__version__}')

MNE 1.10.2  |  NumPy 2.2.6


## 1 — Load and standardise all recordings

In [2]:
STANDARD_CH = [
    'Fp1','Fp2','F7','F3','Fz','F4','F8',
    'T3','C3','Cz','C4','T4',
    'T5','P3','Pz','P4','T6',
    'O1','O2'
]

def load_eeg(filename, target_sfreq=TARGET_SFREQ, l_freq=1.0, h_freq=119.0):
    """
    Load an EDF, keep only the 19 standard 10-20 channels,
    resample if needed, bandpass, average reference.
    """
    path = DATA_DIR / filename
    raw  = mne.io.read_raw_edf(str(path), preload=True, verbose=False)

    # ── channel standardisation ────────────────────────────────────────
    # Build a case-insensitive rename map
    rename = {ch: ch for ch in raw.ch_names
              if ch.upper() in [s.upper() for s in STANDARD_CH]}
    # Correct case to our canonical spelling
    upper_map = {s.upper(): s for s in STANDARD_CH}
    rename = {ch: upper_map[ch.upper()] for ch in raw.ch_names
              if ch.upper() in upper_map}
    raw.rename_channels(rename)
    raw.pick(STANDARD_CH)
    raw.set_channel_types({ch: 'eeg' for ch in STANDARD_CH})
    montage = mne.channels.make_standard_montage('standard_1020')
    raw.set_montage(montage, match_case=False, on_missing='ignore', verbose=False)

    # ── resample if needed ─────────────────────────────────────────────
    if raw.info['sfreq'] != target_sfreq:
        raw.resample(target_sfreq, verbose=False)

    # ── bandpass + average reference ──────────────────────────────────
    h_safe = min(h_freq, target_sfreq / 2 - 1)
    raw.filter(l_freq=l_freq, h_freq=h_safe,
               fir_design='firwin', phase='zero', verbose=False)
    raw.set_eeg_reference('average', projection=True, verbose=False)
    raw.apply_proj()

    return raw

print('Loading recordings...')
raw_awake7   = load_eeg('XUAWAKE7_deidentified.edf')
raw_sleep7   = load_eeg('XUSLEEP7_deidentified.edf')
raw_pre_aw   = load_eeg('XUAWAKEPRE_deidentified.edf')   # awake baseline
raw_pre_sl   = load_eeg('XUSLEEP_deidentified.edf')      # sleep baseline

for name, r in [('AWAKE7', raw_awake7), ('SLEEP7', raw_sleep7),
                ('PRE_awake', raw_pre_aw), ('PRE_sleep', raw_pre_sl)]:
    print(f'  {name:<12}  {r.times[-1]/60:.1f} min  '
          f'{r.info["sfreq"]:.0f} Hz  {len(r.ch_names)} ch')

Loading recordings...


  AWAKE7        9.5 min  256 Hz  19 ch
  SLEEP7        10.6 min  256 Hz  19 ch
  PRE_awake     10.2 min  256 Hz  19 ch
  PRE_sleep     10.2 min  256 Hz  19 ch


## 2 — Shared metrics helpers

In [3]:
BANDS = [
    ('Delta', 0.5,  4.0),
    ('Theta', 4.0,  8.0),
    ('Alpha', 8.0, 13.0),
    ('Beta', 13.0, 30.0),
]

def psd_db(raw, fmax=80.0, n_fft=2048):
    obj = raw.compute_psd(method='welch', fmax=fmax, n_fft=n_fft, verbose=False)
    return obj.freqs, 10 * np.log10(obj.get_data().mean(axis=0) + 1e-30)

def psd_linear(raw, fmax=80.0, n_fft=2048):
    obj = raw.compute_psd(method='welch', fmax=fmax, n_fft=n_fft, verbose=False)
    return obj.freqs, obj.get_data()   # (n_ch, n_freq) V²/Hz

def band_preserve_pct(raw_cleaned, raw_ref, lo, hi, n_fft=2048):
    """% of reference band power retained in cleaned signal."""
    def bp(r):
        obj = r.compute_psd(method='welch', fmin=lo, fmax=hi, n_fft=n_fft, verbose=False)
        return obj.get_data().mean()
    return 100 * bp(raw_cleaned) / (bp(raw_ref) + 1e-30)

def harmonic_atten_db(raw_before, raw_after, h, bw=0.10, n_fft=2048):
    def bp(r):
        obj = r.compute_psd(method='welch', fmin=h-bw, fmax=h+bw, n_fft=n_fft, verbose=False)
        return obj.get_data().mean()
    return 10 * np.log10(bp(raw_after) / (bp(raw_before) + 1e-30))

def apply_dbs_method(raw_dbs, method, **kwargs):
    sfreq = raw_dbs.info['sfreq']
    r = raw_dbs.copy()
    r.load_data()
    d = r.get_data() * 1e6
    d_clean = ArtifactFilterFactory.process(method, d, sfreq, **kwargs)
    r._data = d_clean * 1e-6
    return r

METHODS = [
    ('Spectrum Fit (2Hz)',   'spectrum_fit',         dict(f_target=DBS_FREQ, bandwidth=2.0)),
    ('Comb Notch Q=50',     'comb_notch',            dict(f0=DBS_FREQ, q_factor=50)),
    ('Comb Notch Q=200',    'comb_notch',            dict(f0=DBS_FREQ, q_factor=200)),
    ('Hampel Freq (2Hz)',   'hampel_freq',            dict(window_hz=2.0, n_sigmas=3.0, attenuation_db=-60.0)),
    ('Hampel Time',         'hampel_time',            dict(window_sec=1/DBS_FREQ, n_sigmas=3.0)),
    ('FFT Spectral Interp', 'fft_spectral_interp',   dict(f_target=DBS_FREQ)),
    ('Sinusoidal Regress.', 'sinusoidal_regression', dict(f_target=DBS_FREQ)),
]

PALETTE = [
    ('sienna',     '--', 1.0),
    ('crimson',    '-.', 1.1),
    ('orchid',     '-.', 1.1),
    ('peru',       ':',  1.4),
    ('goldenrod',  ':',  1.4),
    ('steelblue',  '-',  2.2),
    ('darkcyan',   '-',  1.6),
]

print('Helpers ready.')

Helpers ready.


## 3 — 7-method comparison on real data

Run all methods on both AWAKE and SLEEP conditions, compute metrics vs PRE baseline.

### Interpreting the preservation percentages

Because the PRE recording is from a **different session** (no DBS), absolute power levels
differ due to patient state, electrode impedance drift, etc.  Preservation % should be read
as a **relative** measure of brain-band integrity rather than an absolute match:

| % range | Interpretation |
|---------|----------------|
| ~80–120 | Good preservation |
| < 70    | Over-filtered — brain signal lost with the DBS |
| > 150   | DBS harmonic residual inflates that band |

> **Note:** Beta >100% on most methods reflects DBS harmonics at 14, 21, 28 Hz still
> present after DBS removal.  These overlap with beta-band brain activity and are the
> hardest to remove without destroying beta.  FFT Spectral Interpolation is the most
> conservative approach.


In [4]:
# ── Run all 7 methods on AWAKE7 ────────────────────────────────────────
print('=== AWAKE condition ===')
cleaned_awake = {}
for name, mth, kw in METHODS:
    print(f'  {name}')
    cleaned_awake[name] = apply_dbs_method(raw_awake7, mth, **kw)
print('Done.')

=== AWAKE condition ===
  Spectrum Fit (2Hz)
Applying Spectrum-Fit Multi-Harmonic Removal (f=7.0Hz, bandwidth=2.0Hz, attenuation=-60.0dB)
  Comb Notch Q=50
Applying Comb Filter (f0=7.0Hz, Q=50)


  Comb Notch Q=200
Applying Comb Filter (f0=7.0Hz, Q=200)


  Hampel Freq (2Hz)
Running vectorized Freq-Domain Hampel (Allen et al., 2010) - window=2.0Hz, sigmas=3.0, attenuation=-60.0dB


Replaced 25722 frequency domain spikes.
  Hampel Time
Running vectorized Time-Domain Hampel (Allen et al., 2010) - window=37 samples, sigmas=3.0, attenuation=1.0x


  Pass 1: Replaced 208429 outlier points.
  FFT Spectral Interp
FFT Spectral Interpolation: f₀=7.0 Hz, ±2 bins (8.8 mHz per harmonic), 18 harmonics
  → Effective removal: 8.8 mHz per harmonic (18 harmonics, 158.5 mHz total)
  Sinusoidal Regress.
Sinusoidal Regression: f₀=7.0 Hz, 18 harmonics, chunk=full


Done.


In [5]:
# ── Run all 7 methods on SLEEP7 ────────────────────────────────────────
print('=== SLEEP condition ===')
cleaned_sleep = {}
for name, mth, kw in METHODS:
    print(f'  {name}')
    cleaned_sleep[name] = apply_dbs_method(raw_sleep7, mth, **kw)
print('Done.')

=== SLEEP condition ===
  Spectrum Fit (2Hz)
Applying Spectrum-Fit Multi-Harmonic Removal (f=7.0Hz, bandwidth=2.0Hz, attenuation=-60.0dB)
  Comb Notch Q=50
Applying Comb Filter (f0=7.0Hz, Q=50)


  Comb Notch Q=200
Applying Comb Filter (f0=7.0Hz, Q=200)


  Hampel Freq (2Hz)
Running vectorized Freq-Domain Hampel (Allen et al., 2010) - window=2.0Hz, sigmas=3.0, attenuation=-60.0dB


Replaced 45059 frequency domain spikes.
  Hampel Time
Running vectorized Time-Domain Hampel (Allen et al., 2010) - window=37 samples, sigmas=3.0, attenuation=1.0x


  Pass 1: Replaced 217723 outlier points.
  FFT Spectral Interp
FFT Spectral Interpolation: f₀=7.0 Hz, ±2 bins (7.8 mHz per harmonic), 18 harmonics
  → Effective removal: 7.8 mHz per harmonic (18 harmonics, 141.1 mHz total)
  Sinusoidal Regress.
Sinusoidal Regression: f₀=7.0 Hz, 18 harmonics, chunk=full


Done.


In [6]:
# ── Compute metrics for every method × condition ───────────────────────
def compute_metrics(cleaned_dict, raw_dbs, raw_ref, label):
    harmonics = [k * DBS_FREQ for k in range(1, 5)]
    rows = []
    for name, _, _ in METHODS:
        rc = cleaned_dict[name]
        bp  = {b: band_preserve_pct(rc, raw_ref, lo, hi) for b, lo, hi in BANDS}
        att = np.mean([harmonic_atten_db(raw_dbs, rc, h) for h in harmonics])
        rows.append({'name': name, **bp, 'atten': att})
    return rows

metrics_awake = compute_metrics(cleaned_awake, raw_awake7, raw_pre_aw, 'AWAKE')
metrics_sleep = compute_metrics(cleaned_sleep, raw_sleep7, raw_pre_sl, 'SLEEP')

def print_table(rows, title):
    print(f'\n{title}')
    print(f"{'Method':<26} {'Delta%':>7} {'Theta%':>7} {'Alpha%':>7} {'Beta%':>7} {'Atten dB':>9}")
    print('─'*65)
    for r in rows:
        print(f"{r['name']:<26} {r['Delta']:>7.1f} {r['Theta']:>7.1f} "
              f"{r['Alpha']:>7.1f} {r['Beta']:>7.1f} {r['atten']:>9.1f}")

print_table(metrics_awake, '=== AWAKE7 vs PRE_awake ===')
print_table(metrics_sleep, '\n=== SLEEP7 vs PRE_sleep ===')
print('\n100% = matches PRE (brain-only) reference  |  <100% = over-filtered  |  >100% = DBS residual')


=== AWAKE7 vs PRE_awake ===
Method                      Delta%  Theta%  Alpha%   Beta%  Atten dB
─────────────────────────────────────────────────────────────────
Spectrum Fit (2Hz)            34.6    33.6    46.0   125.7     -48.6
Comb Notch Q=50               34.6    75.3    55.2   198.1     -26.5
Comb Notch Q=200              34.6    80.5    55.6   224.7     -15.8
Hampel Freq (2Hz)             32.5    77.8    53.5   227.4     -10.5
Hampel Time                   34.4    68.6    43.6   272.5      -6.4
FFT Spectral Interp           34.6    82.5    55.7   279.1      -5.5
Sinusoidal Regress.           34.6    82.5    55.7   364.5      -3.1


=== SLEEP7 vs PRE_sleep ===
Method                      Delta%  Theta%  Alpha%   Beta%  Atten dB
─────────────────────────────────────────────────────────────────
Spectrum Fit (2Hz)            63.3    25.5   100.2    59.9     -47.1
Comb Notch Q=50               63.3    78.5   123.3   134.3     -27.5
Comb Notch Q=200              63.3    84.7   124.3

In [7]:
# ── PSD comparison — AWAKE7, all 7 methods vs PRE ─────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for ax, (fmin, fmax), title in zip(
    axes,
    [(1, 80), (1, 30)],
    ['Full spectrum 1–80 Hz', 'Zoom 1–30 Hz (critical region)'],
):
    fb, pb = psd_db(raw_pre_aw, fmax=fmax)
    fc, pc = psd_db(raw_awake7, fmax=fmax)
    m = (fb >= fmin) & (fb <= fmax)

    ax.fill_between(fb[m], pb[m]-2, pb[m]+2, color='limegreen', alpha=0.12)
    ax.plot(fb[m], pb[m], color='limegreen', lw=1.2, ls='--', label='PRE baseline')
    ax.plot(fc[m], pc[m], color='gray',      lw=0.7, alpha=0.5, label='DBS contaminated')

    for (name, _, _), (col, ls, lw) in zip(METHODS, PALETTE):
        fn, pn = psd_db(cleaned_awake[name], fmax=fmax)
        mn = (fn >= fmin) & (fn <= fmax)
        ax.plot(fn[mn], pn[mn], color=col, ls=ls, lw=lw, label=name)

    for k in range(1, int(fmax // DBS_FREQ) + 1):
        h = k * DBS_FREQ
        ax.axvline(h, color='orange', lw=0.6, ls=':', alpha=0.6,
                   label='DBS harmonic' if k == 1 else '')

    ax.set_xlabel('Frequency (Hz)'); ax.set_ylabel('PSD (dB)')
    ax.set_title(title); ax.legend(fontsize=7, loc='upper right')

fig.suptitle('XU AWAKE7 — 7-method DBS removal vs PRE baseline',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/real/real_01_awake7_method_comparison.png', dpi=150)
plt.show()

In [8]:
# ── PSD comparison — SLEEP7, all 7 methods vs PRE ─────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for ax, (fmin, fmax), title in zip(
    axes,
    [(1, 80), (1, 30)],
    ['Full spectrum 1–80 Hz', 'Zoom 1–30 Hz (critical region)'],
):
    fb, pb = psd_db(raw_pre_sl, fmax=fmax)
    fc, pc = psd_db(raw_sleep7, fmax=fmax)
    m = (fb >= fmin) & (fb <= fmax)

    ax.fill_between(fb[m], pb[m]-2, pb[m]+2, color='navy', alpha=0.10)
    ax.plot(fb[m], pb[m], color='navy',  lw=1.2, ls='--', label='PRE baseline')
    ax.plot(fc[m], pc[m], color='gray',  lw=0.7, alpha=0.5, label='DBS contaminated')

    for (name, _, _), (col, ls, lw) in zip(METHODS, PALETTE):
        fn, pn = psd_db(cleaned_sleep[name], fmax=fmax)
        mn = (fn >= fmin) & (fn <= fmax)
        ax.plot(fn[mn], pn[mn], color=col, ls=ls, lw=lw, label=name)

    for k in range(1, int(fmax // DBS_FREQ) + 1):
        h = k * DBS_FREQ
        ax.axvline(h, color='orange', lw=0.6, ls=':', alpha=0.6,
                   label='DBS harmonic' if k == 1 else '')

    ax.set_xlabel('Frequency (Hz)'); ax.set_ylabel('PSD (dB)')
    ax.set_title(title); ax.legend(fontsize=7, loc='upper right')

fig.suptitle('XU SLEEP7 — 7-method DBS removal vs PRE baseline',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/real/real_02_sleep7_method_comparison.png', dpi=150)
plt.show()

In [9]:
# ── Master comparison figure: band preservation × methods × conditions ─
method_labels = [r['name'] for r in metrics_awake]
n_methods = len(method_labels)
band_names = [b for b, _, _ in BANDS]
bar_colors = ['sienna','crimson','orchid','peru','goldenrod','steelblue','darkcyan']

fig = plt.figure(figsize=(18, 12))
gs  = gridspec.GridSpec(3, 4, figure=fig, hspace=0.45, wspace=0.35)

# ── Row 0: Band preservation bars — AWAKE ─────────────────────────────
for col_i, bname in enumerate(band_names):
    ax = fig.add_subplot(gs[0, col_i])
    vals = [r[bname] for r in metrics_awake]
    bars = ax.bar(range(n_methods), vals, color=bar_colors, alpha=0.85)
    ax.axhline(100, color='k', lw=1.0, ls='--')
    ax.axhline(90,  color='orange', lw=0.8, ls=':')
    ax.set_title(f'AWAKE — {bname}', fontsize=9, fontweight='bold')
    ax.set_xticks(range(n_methods))
    ax.set_xticklabels([m.split('(')[0].strip() for m in method_labels],
                        rotation=45, ha='right', fontsize=7)
    ax.set_ylabel('Preservation %', fontsize=8)
    ax.set_ylim(40, 200)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, min(val,195)+1, f'{val:.0f}',
                ha='center', va='bottom', fontsize=6, fontweight='bold')

# ── Row 1: Band preservation bars — SLEEP ─────────────────────────────
for col_i, bname in enumerate(band_names):
    ax = fig.add_subplot(gs[1, col_i])
    vals = [r[bname] for r in metrics_sleep]
    bars = ax.bar(range(n_methods), vals, color=bar_colors, alpha=0.85)
    ax.axhline(100, color='k', lw=1.0, ls='--')
    ax.axhline(90,  color='orange', lw=0.8, ls=':')
    ax.set_title(f'SLEEP — {bname}', fontsize=9, fontweight='bold')
    ax.set_xticks(range(n_methods))
    ax.set_xticklabels([m.split('(')[0].strip() for m in method_labels],
                        rotation=45, ha='right', fontsize=7)
    ax.set_ylabel('Preservation %', fontsize=8)
    ax.set_ylim(40, 200)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, min(val,195)+1, f'{val:.0f}',
                ha='center', va='bottom', fontsize=6, fontweight='bold')

# ── Row 2 left: DBS attenuation comparison (awake + sleep) ────────────
ax_att = fig.add_subplot(gs[2, :2])
x = np.arange(n_methods)
w = 0.38
att_aw = [abs(r['atten']) for r in metrics_awake]
att_sl = [abs(r['atten']) for r in metrics_sleep]
bars1 = ax_att.bar(x - w/2, att_aw, w, label='Awake', color=bar_colors, alpha=0.8)
bars2 = ax_att.bar(x + w/2, att_sl, w, label='Sleep', color=bar_colors, alpha=0.5,
                    edgecolor='k', linewidth=0.4)
ax_att.set_xticks(x)
ax_att.set_xticklabels([m.split('(')[0].strip() for m in method_labels],
                         rotation=30, ha='right', fontsize=8)
ax_att.set_ylabel('Mean DBS attenuation |dB|', fontsize=8)
ax_att.set_title('DBS harmonic attenuation (7–28 Hz avg)\n⚠ Wider methods inflate this by removing brain',
                  fontsize=9)
ax_att.legend(fontsize=8)

# ── Row 2 right: Beta preservation summary (both conditions) ──────────
ax_beta = fig.add_subplot(gs[2, 2:])
beta_aw = [r['Beta'] for r in metrics_awake]
beta_sl = [r['Beta'] for r in metrics_sleep]
ax_beta.bar(x - w/2, beta_aw, w, label='Awake', color=bar_colors, alpha=0.8)
ax_beta.bar(x + w/2, beta_sl, w, label='Sleep', color=bar_colors, alpha=0.5,
             edgecolor='k', linewidth=0.4)
ax_beta.axhline(100, color='k',      lw=1.2, ls='--', label='100% = perfect')
ax_beta.axhline(90,  color='orange', lw=0.9, ls=':',  label='90% warning')
ax_beta.set_xticks(x)
ax_beta.set_xticklabels([m.split('(')[0].strip() for m in method_labels],
                          rotation=30, ha='right', fontsize=8)
ax_beta.set_ylabel('Beta preservation %', fontsize=8)
ax_beta.set_title('Brain beta preservation (13–30 Hz)\nMost sensitive to DBS harmonics at 14/21/28 Hz',
                   fontsize=9)
ax_beta.set_ylim(40, 200)
ax_beta.legend(fontsize=8)

fig.suptitle('Real Data (Patient XU, 7 Hz DBS) — Full 7-Method Comparison\n'
             'Preservation % vs PRE baseline  |  100% = no brain lost  |  '
             '<100% = over-filtered  |  >100% = DBS residual',
             fontsize=12, fontweight='bold')
plt.savefig('../figures/real/real_03_master_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Master comparison figure saved.')

Master comparison figure saved.


## 4 — Best method + ICA

Apply FFT Spectral Interpolation (best brain preservation on both synthetic and real data)
followed by ICA to remove eye and muscle artifacts from AWAKE7 and SLEEP7.

In [10]:
def run_ica_pipeline(raw_dbs_removed, label='', n_components=15):
    """
    Fit FastICA, classify eye (Fp1/Fp2 dominant + low-freq) and
    muscle (temporal dominant + high-freq) components, apply.
    """
    sfreq = raw_dbs_removed.info['sfreq']

    ica = mne.preprocessing.ICA(
        n_components=n_components, method='fastica',
        max_iter=800, random_state=42,
    )
    ica.fit(raw_dbs_removed, verbose=False)

    mixing  = ica.get_components()
    sources = ica.get_sources(raw_dbs_removed).get_data()
    ch_lower = [c.lower() for c in raw_dbs_removed.ch_names]
    fp_idx      = [i for i, c in enumerate(ch_lower) if c in ('fp1', 'fp2')]
    muscle_idx  = [i for i, c in enumerate(ch_lower)
                   if c in ('t3', 't4', 't5', 't6', 'f7', 'f8')]

    eye_comps = []; muscle_comps = []
    for ic in range(ica.n_components_):
        col    = np.abs(mixing[:, ic])
        top_ch = int(np.argmax(col))
        f_s, psd_s = signal.welch(sources[ic], fs=sfreq, nperseg=512)
        p_all = np.trapz(psd_s[(f_s>=1)&(f_s<=80)], f_s[(f_s>=1)&(f_s<=80)]) + 1e-30
        lf_r  = np.trapz(psd_s[(f_s>=1)&(f_s<=15)], f_s[(f_s>=1)&(f_s<=15)]) / p_all
        hf_r  = np.trapz(psd_s[(f_s>=30)&(f_s<=80)],f_s[(f_s>=30)&(f_s<=80)]) / p_all
        if top_ch in fp_idx and lf_r > 0.60:
            eye_comps.append(ic)
        elif top_ch in muscle_idx and hf_r > 0.55:
            muscle_comps.append(ic)

    all_bad = sorted(set(eye_comps + muscle_comps))
    print(f'{label}: eye={eye_comps}  muscle={muscle_comps}  → exclude {all_bad}')

    ica.exclude = all_bad
    raw_clean = raw_dbs_removed.copy()
    ica.apply(raw_clean, verbose=False)

    return raw_clean, ica, all_bad

print('ICA function defined.')

ICA function defined.


In [11]:
BEST_METHOD = 'FFT Spectral Interp'

raw_awake_dbs  = cleaned_awake[BEST_METHOD]
raw_sleep_dbs  = cleaned_sleep[BEST_METHOD]

raw_awake_clean, ica_aw, bad_aw = run_ica_pipeline(raw_awake_dbs, label='AWAKE7')
raw_sleep_clean, ica_sl, bad_sl = run_ica_pipeline(raw_sleep_dbs, label='SLEEP7')

AWAKE7: eye=[5]  muscle=[]  → exclude [5]


SLEEP7: eye=[2, 7]  muscle=[]  → exclude [2, 7]


In [12]:
fig = plt.figure(figsize=(16, 8))
gs_ica = gridspec.GridSpec(2, 4, figure=fig, hspace=0.45, wspace=0.35)

for row_i, (ica_obj, bad, raw_obj, label) in enumerate([
    (ica_aw, bad_aw, raw_awake_dbs, 'AWAKE7'),
    (ica_sl, bad_sl, raw_sleep_dbs, 'SLEEP7'),
]):
    # Topomap of excluded components (up to 3)
    n_show = min(len(bad), 3)
    mixing = ica_obj.get_components()   # shape (n_ch, n_components)
    for col_i in range(3):
        ax = fig.add_subplot(gs_ica[row_i, col_i])
        if col_i < n_show:
            ic  = bad[col_i]
            mix = mixing[:, ic]
            mne.viz.plot_topomap(mix, raw_obj.info, axes=ax, show=False, cmap='RdBu_r',
                                 vlim=(-np.abs(mix).max(), np.abs(mix).max()))
            ax.set_title(f'{label}\nIC{ic} (excluded)', fontsize=8)
        else:
            ax.axis('off')

    # PSD: contaminated → DBS removed → final clean
    ax_psd = fig.add_subplot(gs_ica[row_i, 3])
    raw_dbs_cond = raw_awake7 if 'AWAKE' in label else raw_sleep7
    raw_pre_cond = raw_pre_aw if 'AWAKE' in label else raw_pre_sl
    raw_final    = raw_awake_clean if 'AWAKE' in label else raw_sleep_clean

    for raw_s, col, lbl_s in [
        (raw_dbs_cond, 'gray',      'DBS contaminated'),
        (raw_obj,      'steelblue', 'DBS removed'),
        (raw_final,    'darkblue',  'Final (+ ICA)'),
        (raw_pre_cond, 'limegreen', 'PRE baseline'),
    ]:
        f_, p_ = psd_db(raw_s, fmax=30.0)
        m_ = (f_ >= 1) & (f_ <= 30)
        ax_psd.plot(f_[m_], p_[m_], color=col,
                    lw=2.0 if 'Final' in lbl_s else 1.0,
                    ls='--' if 'PRE' in lbl_s else '-', label=lbl_s)
    for k in range(1, 5):
        ax_psd.axvline(k * DBS_FREQ, color='orange', lw=0.7, ls=':', alpha=0.6)
    ax_psd.set_xlabel('Hz'); ax_psd.set_ylabel('PSD (dB)')
    ax_psd.set_title(f'{label} — pipeline stages', fontsize=9)
    ax_psd.legend(fontsize=7)

fig.suptitle('ICA artifact removal — AWAKE7 and SLEEP7', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/real/real_04_ica_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('ICA results figure saved.')


ICA results figure saved.


## 5 — Final comprehensive summary

In [13]:
# ── 4-panel final PSD: awake and sleep, full + zoom ────────────────────
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

for row_i, (raw_dbs_cond, raw_dbs_rem, raw_final, raw_ref, cond_label, ref_col) in enumerate([
    (raw_awake7, raw_awake_dbs, raw_awake_clean, raw_pre_aw, 'AWAKE7', 'limegreen'),
    (raw_sleep7, raw_sleep_dbs, raw_sleep_clean, raw_pre_sl, 'SLEEP7', 'navy'),
]):
    for col_i, (fmin, fmax) in enumerate([(1, 80), (1, 30)]):
        ax = axes[row_i, col_i]
        for raw_s, col, lw, ls, lbl in [
            (raw_dbs_cond, 'gray',     0.7, '-',  'DBS contaminated'),
            (raw_dbs_rem,  'steelblue',1.2, '-',  f'DBS removed ({BEST_METHOD})'),
            (raw_final,    'darkblue', 2.0, '-',  'Final clean (+ ICA)'),
            (raw_ref,      ref_col,    1.0, '--', 'PRE baseline'),
        ]:
            f_, p_ = psd_db(raw_s, fmax=fmax)
            m_ = (f_ >= fmin) & (f_ <= fmax)
            ax.plot(f_[m_], p_[m_], color=col, lw=lw, ls=ls, alpha=0.85, label=lbl)

        if fmax <= 30:
            for k in range(1, 5):
                h = k * DBS_FREQ
                if h <= fmax:
                    ax.axvline(h, color='orange', lw=0.9, ls=':', alpha=0.7,
                               label='DBS harmonic' if k == 1 else '')
        ax.set_xlabel('Frequency (Hz)'); ax.set_ylabel('PSD (dB)')
        suffix = 'Full 1–80 Hz' if fmax == 80 else 'Zoom 1–30 Hz'
        ax.set_title(f'{cond_label} — {suffix}', fontsize=10)
        ax.legend(fontsize=7)

fig.suptitle('XU Patient — Final Pipeline: DBS Removal + ICA\n'
             f'Method: {BEST_METHOD}  |  Both AWAKE and SLEEP conditions',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/real/real_05_final_pipeline_psd.png', dpi=150)
plt.show()

In [14]:
def topomap_band(raw, lo, hi, ax, title):
    obj   = raw.compute_psd(method='welch', fmin=lo, fmax=hi, n_fft=2048, verbose=False)
    bp    = obj.get_data().mean(axis=1)
    bp_db = 10 * np.log10(bp + 1e-30)
    mne.viz.plot_topomap(
        bp_db, raw.info, axes=ax, show=False, cmap='RdBu_r',
        vlim=(np.percentile(bp_db, 5), np.percentile(bp_db, 95)),
    )
    ax.set_title(title, fontsize=8)

awake_stages = [
    (raw_awake7,      'AWAKE\nContaminated'),
    (raw_awake_dbs,   'AWAKE\nDBS removed'),
    (raw_awake_clean, 'AWAKE\nFinal clean'),
    (raw_pre_aw,      'AWAKE\nPRE baseline'),
]
sleep_stages = [
    (raw_sleep7,      'SLEEP\nContaminated'),
    (raw_sleep_dbs,   'SLEEP\nDBS removed'),
    (raw_sleep_clean, 'SLEEP\nFinal clean'),
    (raw_pre_sl,      'SLEEP\nPRE baseline'),
]

fig, axes = plt.subplots(4, 4, figsize=(16, 14))
for col_i, (raw_s, lbl) in enumerate(awake_stages):
    topomap_band(raw_s, 4,  8, axes[0, col_i], f'{lbl}\nTheta 4–8 Hz')
    topomap_band(raw_s, 8, 13, axes[1, col_i], f'{lbl}\nAlpha 8–13 Hz')
for col_i, (raw_s, lbl) in enumerate(sleep_stages):
    topomap_band(raw_s, 4,  8, axes[2, col_i], f'{lbl}\nTheta 4–8 Hz')
    topomap_band(raw_s, 8, 13, axes[3, col_i], f'{lbl}\nAlpha 8–13 Hz')

fig.suptitle('Band power topomaps — pipeline stages (AWAKE + SLEEP)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/real/real_06_topomaps.png', dpi=150, bbox_inches='tight')
plt.show()
print('Topomaps figure saved.')


Topomaps figure saved.


In [15]:
# ── Quantitative final summary ─────────────────────────────────────────
def full_summary(raw_dbs, raw_cleaned, raw_ref, label):
    f_d, p_d  = psd_linear(raw_dbs)
    f_cl, p_cl = psd_linear(raw_cleaned)
    f_r, p_r  = psd_linear(raw_ref)

    def bp(freqs, psd, lo, hi):
        m = (freqs >= lo) & (freqs <= hi)
        return np.trapz(psd[:, m], freqs[m], axis=1).mean()

    print(f'\n{"═"*80}')
    print(f'{label} — Band power summary (dB re 1 V²/Hz, mean across channels)')
    print(f'{"═"*80}')
    print(f"{'Band':<20} {'PRE baseline':>13} {'DBS contam.':>13} {'Final clean':>13} {'Preserved%':>12}")
    print(f"{'─'*75}")
    for bname, lo, hi in BANDS:
        b_  = bp(f_r,  p_r,  lo, hi)
        d_  = bp(f_d,  p_d,  lo, hi)
        cl_ = bp(f_cl, p_cl, lo, hi)
        pct = 100 * cl_ / (b_ + 1e-30)
        print(f'{bname:<20} {10*np.log10(b_+1e-30):>13.1f} '
              f'{10*np.log10(d_+1e-30):>13.1f} '
              f'{10*np.log10(cl_+1e-30):>13.1f} {pct:>11.1f}%')

    print(f'\nDBS harmonic attenuation — contaminated → final clean:')
    for k in range(1, 8):
        h = k * DBS_FREQ
        if h >= raw_dbs.info['sfreq'] / 2: break
        a = harmonic_atten_db(raw_dbs, raw_cleaned, h)
        print(f'  k={k}  {h:5.1f} Hz  →  {a:+.1f} dB')

full_summary(raw_awake7, raw_awake_clean, raw_pre_aw, 'XU AWAKE7')
full_summary(raw_sleep7, raw_sleep_clean, raw_pre_sl, 'XU SLEEP7')


════════════════════════════════════════════════════════════════════════════════
XU AWAKE7 — Band power summary (dB re 1 V²/Hz, mean across channels)
════════════════════════════════════════════════════════════════════════════════
Band                  PRE baseline   DBS contam.   Final clean   Preserved%
───────────────────────────────────────────────────────────────────────────
Delta                        -95.4        -100.1        -100.7        29.3%
Theta                       -100.7        -101.5        -101.6        81.3%
Alpha                       -102.5        -105.0        -105.0        55.5%
Beta                        -108.4        -100.9        -104.0       276.9%

DBS harmonic attenuation — contaminated → final clean:
  k=1    7.0 Hz  →  -0.5 dB
  k=2   14.0 Hz  →  -9.2 dB
  k=3   21.0 Hz  →  -4.8 dB
  k=4   28.0 Hz  →  -7.5 dB


  k=5   35.0 Hz  →  -5.4 dB
  k=6   42.0 Hz  →  -8.9 dB


  k=7   49.0 Hz  →  -10.3 dB

════════════════════════════════════════════════════════════════════════════════
XU SLEEP7 — Band power summary (dB re 1 V²/Hz, mean across channels)
════════════════════════════════════════════════════════════════════════════════
Band                  PRE baseline   DBS contam.   Final clean   Preserved%
───────────────────────────────────────────────────────────────────────────
Delta                        -95.4         -97.4         -98.6        48.1%
Theta                       -100.7        -101.3        -102.0        73.1%
Alpha                       -102.5        -101.6        -102.1       110.4%
Beta                        -108.4        -101.2        -105.4       200.1%

DBS harmonic attenuation — contaminated → final clean:
  k=1    7.0 Hz  →  -1.6 dB
  k=2   14.0 Hz  →  -8.1 dB


  k=3   21.0 Hz  →  -4.4 dB


  k=4   28.0 Hz  →  -7.6 dB
  k=5   35.0 Hz  →  -8.8 dB
  k=6   42.0 Hz  →  -12.2 dB
  k=7   49.0 Hz  →  -21.7 dB
